# Vibrational Spectroscopy: FTIR & Raman Analysis

This notebook demonstrates quantitative vibrational spectroscopy for catalyst characterization:
- FTIR transmittance to absorbance conversion via Beer-Lambert: $A = -\log_{10}(T)$
- Inverted descending wavenumber axis ($4000 \to 400\text{ cm}^{-1}$)
- Characteristic functional group region mapping
- Raman cosmic-ray spike rejection & AsLS baseline subtraction
- Raman D-band and G-band deconvolution with explicit distinction between:
  - Peak-height ratio ($I_D / I_G$)
  - Integrated peak-area ratio ($A_D / A_G$).

In [1]:
import sys
from pathlib import Path
root_path = Path.cwd().parent.parent if Path.cwd().name == 'characterization' else Path.cwd()
if str(root_path) not in sys.path:
    sys.path.insert(0, str(root_path))

import matplotlib.pyplot as plt
import numpy as np
from parsers.characterization import parse_ftir_file, convert_transmittance_to_absorbance, parse_raman_file
from modules.characterization.ftir import plot_ftir_spectrum, plot_ftir_stacked, analyze_ftir_bands
from modules.characterization.raman import (
    process_raman_spectrum,
    calculate_peak_ratios,
    plot_raman_spectrum,
    plot_raman_stacked,
    plot_raman_deconvolution,
)
from modules.characterization.common_peaks import fit_peaks
from scripts.utils import apply_publication_style

apply_publication_style()
print('Spectroscopy modules loaded.')

Spectroscopy modules loaded.


## 1. FTIR Surface Functionalization & Band Identification
Load catalyst FTIR transmittance and convert to absorbance.

In [2]:
ftir_path = root_path / 'data' / 'examples' / 'characterization' / 'ftir' / 'catalyst_functionalized_ftir.csv'
raw_ftir = parse_ftir_file(ftir_path)
abs_ftir = convert_transmittance_to_absorbance(raw_ftir)

fig, ax = plot_ftir_spectrum(abs_ftir, annotate_peaks=True, show_regions=True, title='Functionalized Catalyst Surface FTIR')
plt.show()

C:\Users\mjram\python_graphing\modules\characterization\ftir.py:139: UserWarning: The figure layout has changed to tight
  fig.tight_layout()
C:\Users\mjram\AppData\Local\Temp\ipykernel_22912\3722226810.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. FTIR Degradation Comparison: Pristine vs Spent
Examine loss of surface ligands following electrochemical cycling.

In [3]:
spent_path = root_path / 'data' / 'examples' / 'characterization' / 'ftir' / 'catalyst_spent_ftir.csv'
spent_ftir = convert_transmittance_to_absorbance(parse_ftir_file(spent_path))

fig, ax = plot_ftir_stacked([abs_ftir, spent_ftir], labels=['Pristine', 'Spent Post-OER'], vertical_offset_fraction=0.4, title='FTIR Surface Degradation')
plt.show()

C:\Users\mjram\python_graphing\modules\characterization\ftir.py:195: UserWarning: The figure layout has changed to tight
  fig.tight_layout()
C:\Users\mjram\AppData\Local\Temp\ipykernel_22912\911031182.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Raman Spectroscopy: Defect Density & D/G Ratios
Deconvolute carbon black support D-band (~1345 cm^-1) and G-band (~1585 cm^-1).

In [4]:
raman_path = root_path / 'data' / 'examples' / 'characterization' / 'raman' / 'carbon_support_raman.csv'
raw_raman = parse_raman_file(raman_path)
proc_raman = process_raman_spectrum(raw_raman, despike=True, subtract_bg=True, bg_method='asls')

fit_res = fit_peaks(proc_raman.x, proc_raman.y, peak_centers=[1345.0, 1585.0], model_type='lorentzian', background='linear')
ratios = calculate_peak_ratios(fit_res, peak_1_index=1, peak_2_index=2, label_1='D', label_2='G')
print(f'Peak Height Ratio (I_D / I_G): {ratios["height_ratio"]:.3f}')
print(f'Integrated Area Ratio (A_D / A_G): {ratios["area_ratio"]:.3f}')

fig, axes = plot_raman_deconvolution(proc_raman, fit_res, title='Carbon Support D-Band & G-Band Deconvolution')
plt.show()

Peak Height Ratio (I_D / I_G): 0.678
Integrated Area Ratio (A_D / A_G): 0.794


C:\Users\mjram\python_graphing\modules\characterization\raman.py:253: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
C:\Users\mjram\python_graphing\modules\characterization\raman.py:253: UserWarning: The figure layout has changed to tight
  fig.tight_layout()
C:\Users\mjram\AppData\Local\Temp\ipykernel_22912\2638395705.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
